In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

Matplotlib is building the font cache; this may take a moment.


In [ ]:
#### INPUT PATH AND LOAD DATA
INPUT_FOLDER = 'alex/force_dist_array_fullrod/output'
r0_values = [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 16, 18, 20, 22, 24, 26, 28, 30]
k_values = [0.35, 0.45]


dfs = {}  # dfs[k][r0] = dataframe

for k in k_values:
    k_tag = f"{int(round(k * 100)):03d}"   # 0.35 -> "035", 0.45 -> "045"
    dfs[k] = {}

    for r0 in r0_values:
        folder = f"K_{k_tag}_r0_{r0}"
        run    = f"K_{k_tag}_r0_{r0}_traps_MD_long"
        path   = os.path.join(INPUT_FOLDER, folder, run, "obs.dat")

        dfs[k][r0] = pd.read_csv(
            path,
            sep=r"\s+",          
            header=None,
            names=["step", "dist", "force_mag", "hbond"]
        )

In [ ]:
CUTOFF = 0.25e7  # 2,500,000

dfs_cfiltered = {}  # dfs_cfiltered[k][r0] = cutoff-filtered df

for k, by_r0 in dfs.items():
    dfs_cfiltered[k] = {}
    for r0, df in by_r0.items():
        dfs_cfiltered[k][r0] = df[df["step"] >= CUTOFF].reset_index(drop=True)

In [ ]:
import matplotlib.pyplot as plt

r0_values = [2,3,4,5,6,7,8,9,10,11,12,13,14,16,18,20,22,24,26,28,30]
k_values = [0.35, 0.45]

for k in k_values:
    fig, ax = plt.subplots(figsize=(8,6), dpi=500)

    if k not in dfs:
        continue

    for r0 in r0_values:
        if r0 not in dfs[k]:
            continue
        df = dfs[k][r0]
        ax.plot(df["step"], df["hbond"],
                alpha=0.9, linewidth=0.5,
                label=f"r0={r0}")

    ax.set_title(f"H-bonds vs time (K={k})", fontsize=14)
    ax.set_xlabel("EPOCHS", fontsize=14)
    ax.set_ylabel("Number of H-Bonds", fontsize=14)

    # With many curves, legend can get huge; you can comment it out or shrink it:
    ax.legend(fontsize=8, ncol=2, frameon=True)

    plt.show()

In [ ]:
def steady_cutoff_by_hbonds(df, *,
                            tail_frac=0.10,
                            band=2.0,
                            min_stay=0.1e7,
                            ):
    """
    Returns cutoff step where steady state begins, based on hbond convergence.
    df must have columns: 'step', 'hbond' (sorted by step).
    """
    d = df[["step", "hbond"]].dropna().sort_values("step").reset_index(drop=True)
    n = len(d)
    if n == 0:
        return None

    tail_n = int(n * tail_frac)
    plateau = d["hbond"].iloc[-tail_n:].median()  # robust plateau estimate

    in_band = (d["hbond"] >= plateau - band) & (d["hbond"] <= plateau + band)

    # Find earliest index i such that stable is True for the next min_stay points
    stable_run = in_band.rolling(min_stay, min_periods=min_stay).sum() >= min_stay
    idx = stable_run.idxmax() if stable_run.any() else None

    if idx is None:
        # fallback: if never "stays", cut at start of tail window
        return int(d["step"].iloc[-tail_n])

    # rolling window marks the END of the first full stable block; shift back
    start_idx = max(0, idx - (min_stay - 1))
    return int(d["step"].iloc[start_idx])

In [ ]:
import matplotlib.pyplot as plt

source = dfs

# Parameters for cutoff detection
TAIL_FRAC = 0.10
BAND = 2.0
MIN_STAY_STEPS = 0.1e7  # adjust (e.g. 1e6 or 2e6)

cutoffs = {}      # cutoffs[k][r0] = cutoff_step
dfs_ss = {}       # steady-state cropped dfs: dfs_ss[k][r0]

for k, by_r0 in source.items():
    cutoffs[k] = {}
    dfs_ss[k] = {}
    for r0, df in by_r0.items():
        c = steady_cutoff_by_hbonds(
            df,
            tail_frac=TAIL_FRAC,
            band=BAND,
            min_stay_steps=MIN_STAY_STEPS,
        )
        cutoffs[k][r0] = c
        dfs_ss[k][r0] = df[df["step"] >= c].reset_index(drop=True)

# ---- Plot hbonds time series with vertical cutoff lines (one figure per k) ----
for k, by_r0 in source.items():
    fig, ax = plt.subplots(figsize=(8, 6), dpi=500)

    for r0, df in by_r0.items():
        ax.plot(df["step"], df["hbond"], linewidth=0.5, alpha=0.9, label=f"r0={r0}")

        c = cutoffs[k][r0]
        ax.axvline(c, linestyle="--", linewidth=0.7, alpha=0.6)

    ax.set_title(f"H-bonds vs time with detected cutoff (K={k})", fontsize=14)
    ax.set_xlabel("EPOCHS", fontsize=14)
    ax.set_ylabel("Number of H-Bonds", fontsize=14)

    
    ax.legend(fontsize=7, ncol=2, frameon=True)

    plt.tight_layout()
    plt.show()

In [ ]:
#### BASIC STATISTICS OF STEADY STATE REGION
import numpy as np
import pandas as pd

r0_values = [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 16, 18, 20, 22, 24, 26, 28, 30]

stats_data = []
for k, by_r0 in dfs_cfiltered.items():
    for r0 in r0_values:
        if r0 not in by_r0:
            continue

        df = by_r0[r0]

        # signed force: +|F| if dist > r0, -|F| if dist < r0
        # np.sign gives -1, 0, +1; equality -> 0 (force becomes 0 at exact r0)
        signed_force = np.sign(df["dist"].to_numpy() - r0) * df["force_mag"].to_numpy()

        stats_data.append({
            "k": k,
            "r_0": float(r0),

            "F mean": signed_force.mean(),
            "F std": signed_force.std(ddof=0),

            "|F| mean": df["force_mag"].mean(),
            "|F| std": df["force_mag"].std(ddof=0),

            "dist mean": df["dist"].mean(),
            "dist std": df["dist"].std(ddof=0),
            "hbonds mean": df["hbond"].mean(),
        })

stats_df = pd.DataFrame(stats_data).sort_values(["k", "r_0"])
print(stats_df)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8,6), dpi=500)

for k, label in [(0.35, "K=0.35"), (0.45, "K=0.45")]:
    sub = stats_df[stats_df["k"] == k].sort_values("r0")
    ax.plot(sub["r0"], sub["|F| mean"], marker="o", label=label)

ax.set_xlabel(r"Equilibrium Distance r$_0$ (oxDNA units)", fontsize=14)
ax.set_ylabel("Mean Force Magnitude (oxDNA units)", fontsize=14)
ax.legend(fontsize=12)

plt.show()